# Benchmarking `spherical.jaxwigner` on a Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucymthomas/spherical/blob/jax-wigner/benchmarks/colab_benchmark.ipynb)

This notebook runs `benchmarks/gpu_benchmark.py` on a Google Colab GPU.  It is the Colab
version of the cluster instructions in `benchmarks/README.md`, and produces the same
JSON/CSV results.

**Before running anything:** *Runtime → Change runtime type → Hardware accelerator: GPU*
(T4 on the free tier; L4 or A100 with Colab Pro), then *Save*.

Then run the cells from top to bottom.  Steps 1–4 take a few minutes; step 6 is the
long one.

**Colab caveats**
- Sessions disconnect after a period of inactivity, and free sessions have a time limit.
  Saving results to Google Drive (step 3) means a disconnect loses at most the
  operation being timed: the script saves after every operation.
- Colab machines have only a couple of CPU cores, so **compile times** here are longer
  than on a cluster node.  The run times (`median_s`) are what measure the GPU.
- The GPU you get varies between sessions.  The JSON records which one it was.
- Free-tier GPUs (T4) run float64 far more slowly than float32.


## 1. Check the GPU

In [ ]:
!nvidia-smi

import jax
print("JAX", jax.__version__, "devices:", jax.devices())
assert jax.default_backend() == "gpu", "No GPU: choose a GPU runtime (Runtime → Change runtime type) and rerun."


## 2. Get the code and install it

Colab already has a GPU build of JAX matching its CUDA libraries, so this installs
`spherical` **without** reinstalling JAX.  Change `BRANCH` to test another branch.

In [ ]:
REPO = "https://github.com/lucymthomas/spherical.git"
BRANCH = "jax-wigner"

import os
if not os.path.exists("/content/spherical"):
    !git clone --branch {BRANCH} {REPO} /content/spherical
else:
    !git -C /content/spherical fetch origin && git -C /content/spherical checkout {BRANCH} && git -C /content/spherical pull --ff-only
%cd /content/spherical
!git log --oneline -1
!pip install --quiet -e . matplotlib


In [ ]:
# Check that it imports and agrees with the numba implementation on the GPU
import subprocess, sys
print(subprocess.run([sys.executable, "-c", '''
import jax; jax.config.update("jax_enable_x64", True)
import numpy as np, quaternionic, spherical
from spherical import jaxwigner as jw
R = quaternionic.array(np.random.default_rng(1).normal(size=(100, 4))).normalized
D_jax = np.asarray(jw.wigner_D(jax.numpy.asarray(R.ndarray), 8))
D_ref = spherical.Wigner(8).D(R)
print("device:", jax.devices()[0], "  max |JAX - numba| =", np.max(np.abs(D_jax - D_ref)))
'''], capture_output=True, text=True).stdout or "import failed: rerun the install cell")


## 3. Where to save results

Saving to Google Drive keeps results if the session disconnects.  Running this cell asks
for permission to access your Drive.  To keep results only in the session (lost when it
ends), set `USE_DRIVE = False`.

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/jaxwigner_benchmarks"
else:
    OUT = "/content/benchmark_results"
os.makedirs(OUT, exist_ok=True)

# A short tag naming the GPU, used in the file names, e.g. "colab-Tesla_T4"
GPU = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip().splitlines()[0]
TAG = "colab-" + GPU.replace(" ", "_")
print("Saving to", OUT, "with tag", TAG)


## 4. Quick run (a few minutes)

ℓ_max = 8, N up to 10⁶, the five main operations.  Checks everything works and gives a
first look.

In [ ]:
!python benchmarks/gpu_benchmark.py --quick --tag {TAG}-quick --output-dir {OUT} --plot


## 5. Look at the results

In [ ]:
import glob, json
import pandas as pd
from IPython.display import Image, display

def latest(pattern=""):
    files = sorted(glob.glob(f"{OUT}/jaxwigner_benchmark_*{pattern}*.json"), key=os.path.getmtime)
    return files[-1] if files else None

def summarise(path):
    with open(path) as f:
        results = json.load(f)
    print(os.path.basename(path), "| complete:", results.get("complete", True),
          "|", results["environment"].get("device_used"))
    for a in results["accuracy"]:
        print("  accuracy:", a)
    t = pd.DataFrame(results["timings"])
    ok = t[(t.status == "ok") & (t.implementation == "jax")]
    table = ok.pivot_table(index=["precision", "ell_max", "op"], columns="N", values="ns_per_rotor")
    display(table.style.format("{:.2f}", na_rep="").set_caption("ns per rotor (median)"))
    failed = t[t.status != "ok"]
    if len(failed):
        display(failed[["op", "ell_max", "precision", "N", "status"]])
    for png in sorted(glob.glob(os.path.splitext(path)[0] + "_*.png")):
        display(Image(png))

summarise(latest("quick"))


## 6. Main run

The full default run (ℓ_max ∈ {2, 4, 8}, N up to 1.6×10⁷, 9 operations, float64 and
float32) is too long for a free Colab session.  These presets split it up; pick one by
setting `PRESET`, and run the others in later sessions if you like.

| preset | what it runs |
|---|---|
| `"l8"` | ℓ_max = 8, all operations, both precisions, N up to 4×10⁶. **Start here.** |
| `"float32"` | ℓ_max ∈ {2, 4, 8}, float32 only, all operations, all N |
| `"float64"` | ℓ_max ∈ {2, 4, 8}, float64 only, all operations, all N (slow on a T4) |
| `"full"` | the whole default benchmark, as on the cluster |

Progress is printed as each measurement finishes.  If the session disconnects, the
results so far are already saved; rerun cells 2 and 3, then step 5 with
`summarise(latest())` to see them.

In [ ]:
PRESET = "l8"

PRESETS = {
    "l8":      "--ell-max 8 --sizes 1000 10000 100000 1000000 4000000",
    "float32": "--precision float32",
    "float64": "--precision float64",
    "full":    "",
}
ARGS = PRESETS[PRESET]
!python benchmarks/gpu_benchmark.py {ARGS} --tag {TAG}-{PRESET} --output-dir {OUT} --plot


In [ ]:
summarise(latest(PRESET))


## 7. Get the results

If you used Drive, the files are in **My Drive → jaxwigner_benchmarks**.  Otherwise this
cell downloads the newest results to your computer.  The `.json` file has everything
(environment, accuracy and every timing).

In [ ]:
from google.colab import files
path = latest()
print(path)
files.download(path)


## Optional: same machine, CPU only

For a CPU-against-GPU comparison on the same Colab machine (only two cores, so this says
little about a real CPU node):

In [ ]:
!python benchmarks/gpu_benchmark.py --device cpu --quick --skip-accuracy --tag colab-cpu --output-dir {OUT}
